# Model 1 · Utterance scorer for extractive dialogue notes

The whole Model 1 pipeline in one self-contained notebook:
1. DialogSum + WikiText-103, the shared WordPiece tokenizer (reused from an attached Model 2
   output if there is one, else trained here with the same deterministic recipe)
2. Teacher labels from **Qwen2.5-7B-Instruct-AWQ** and **Qwen2.5-1.5B-Instruct** (vLLM, greedy).
   The teachers run in **their own virtual environment** as a subprocess, before JAX touches
   the GPU; the training environment never imports vLLM or PyTorch.
3. Oracle labels (greedy ROUGE-1 + ROUGE-2)
4. MLM pretraining of the ~17M-parameter encoder (3 h step budget from a throughput probe)
5. Students A (7B), B (1.5B), C (oracle), D (7B, no MLM), E (oracle importance + 7B
   tags), 3 seeds each
6. A preview results table (ROUGE, retention, agreement, tag F1). BERTScore and CPU latency
   are added by the repo's evaluation after the outputs are fetched.

**Run**: Settings → Accelerator **GPU T4 x2**, Internet **on** → Run All. It is sized for
one session (~7–9 h); if a session is cut short, it stops cleanly before 11 h. **Resume**: *Add Input → Notebook Output* →
this notebook's previous version, then Run All; finished phases, teacher labels, epochs and
seeds are kept. Optionally also attach the Model 2 notebook output to reuse its tokenizer.

**Student E only** (A–D already trained): attach the previous output and Run All; labels,
encoder and finished students are restored and skipped, so only E trains (~2–3 h).

**Output** (`/kaggle/working`): `data/labels/{qwen7b,qwen1_5b,oracle}/`,
`checkpoints/mlm/encoder.keras`, `checkpoints/model1_student_*/seed_<n>/best.keras`,
`checkpoints/model1_tokenizer/`, `results/<run>/seed_<n>/` (metrics, history, test
predictions), `results/tables/model1_preview.md`, `manifest_model1_scorer.json`.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # before any keras import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import time
from pathlib import Path

SESSION_START = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = os.environ.get("SMOKE") == "1"  # tiny CPU run with a fake teacher (< 2-3 min)
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
INPUT = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
STAGE = "model1_scorer"
print(f"on_kaggle={ON_KAGGLE} smoke={SMOKE} work={WORK}")

In [ ]:
# Exact versions for the training environment (the teachers get their own venv below).
import subprocess

if ON_KAGGLE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.10.2", "keras==3.15.1",
         "datasets==5.1.0", "tokenizers==0.23.3", "rouge-score==0.1.2", "scipy"],
        check=True,
    )

## Configuration
Every hyperparameter of the run is here.

In [ ]:
MODEL = {"vocab_size": 16000, "max_len": 512, "d_model": 384, "num_layers": 6, "num_heads": 6,
         "ffn_dim": 1536, "dropout": 0.1, "init_std": 0.02, "num_speakers": 8}
CONFIG = {
    "seed": 0,
    "max_hours": 11,                     # per Kaggle session (12 h cap), every phase included
    "min_hours_to_start": 0.25,          # do not start MLM / a student seed with less time left
    "wikitext": {"hf_path": "Salesforce/wikitext", "hf_config": "wikitext-103-raw-v1",
                 "revision": "b08601e04326c79dfdd32d625aee71d232d685c3"},
    "dialogsum": {"hf_path": "knkarthick/dialogsum",
                  "revision": "a968e7aee0602e257935f1321a02e4287f7d5848", "max_speakers": 8},
    "max_units": None,                   # cap on articles / dialogues per split (smoke only)
    "tokenizer": {"vocab_size": 16000, "min_frequency": 2, "limit_alphabet": 1000,
                  "wordpieces_prefix": "##", "max_items_per_source": None},
    "teacher": {
        "models": {"qwen7b": "Qwen/Qwen2.5-7B-Instruct-AWQ",
                   "qwen1_5b": "Qwen/Qwen2.5-1.5B-Instruct"},
        "fallback_models": {"qwen7b": "Qwen/Qwen2.5-7B-Instruct",
                            "qwen1_5b": "Qwen/Qwen2.5-1.5B-Instruct"},
        "vllm_spec": "vllm",             # pin (e.g. vllm==X.Y.Z) once a run has worked
        "python": "3.12",                # teacher venv (uv): best vLLM / torch wheel support
        "temperature": 0.0,
        "max_new_tokens": 2048,
        "max_model_len": 6144,
        "gpu_memory_utilization": 0.9,
        "enable_prefix_caching": True,   # system prompt + instructions shared by every request
        "score_range": [0, 10],
        "tags": ["none", "key_point", "decision", "action_item"],
        "splits": ["train", "validation", "test"],
        "train_subset": None,            # PLAN fallback: 5000 (same subset for both teachers)
        "batch_dialogues": 256,
        "fallback_batch_size": 8,
    },
    "oracle": {"rouge_types": ["rouge1", "rouge2"], "max_utterances": 3, "use_stemmer": True},
    "mlm": {
        "name": "model1_mlm",
        "seq_len": 512,
        "corpus": {"wikitext_train": 1, "dialogsum_train": 5},  # value = copies
        "held_out": ["wikitext_validation", "dialogsum_validation"],
        "mask_prob": 0.15, "mask_token_frac": 0.8, "random_token_frac": 0.1,
        "optimizer": {"peak_lr": 5.0e-4, "warmup_steps": 4000, "weight_decay": 0.01},
        "batch_size": 64,
        "max_steps": None,               # None = throughput probe -> target_hours
        "target_hours": 3,               # PLAN: ~8-10 GPU-hours; 3 approved (~8 corpus passes)
        "probe_steps": 50,
        "steps_per_epoch": 1000,         # checkpoint / held-out evaluation interval
        "mixed_precision": "mixed_float16",
        "steps_per_execution": 32,       # throughput only: host syncs once per 32 steps
        "workers": 4,                    # batch-prefetch threads
    },
    "students": {                        # name -> targets, tag head, init, checkpoint dir
        "model1_student_A": {"source": "qwen7b", "use_tags": True, "init": "mlm",
                             "dir": "model1_student_A"},
        "model1_student_B": {"source": "qwen1_5b", "use_tags": True, "init": "mlm",
                             "dir": "model1_student_B"},
        "model1_student_C": {"source": "oracle", "use_tags": False, "init": "mlm",
                             "dir": "model1_student_C"},
        "model1_student_D_noMLM": {"source": "qwen7b", "use_tags": True, "init": "random",
                                   "dir": "model1_student_D"},
        # Importance from the oracle, tags from the 7B teacher: A copies the teacher's bias
        # towards late, long utterances; the oracle is what the human summaries use.
        "model1_student_E": {"source": "oracle", "tags_from": "qwen7b", "use_tags": True,
                             "init": "mlm", "dir": "model1_student_E"},
    },
    "student": {
        "seeds": [0, 1, 2],
        "max_utterances": 80,
        "loss": {"listwise_lambda": 1.0, "temperature": 2.0, "tag_weight": 0.5,
                 "score_scale": 10},
        "optimizer": {"lr_encoder": 1.0e-4, "lr_heads": 1.0e-3, "warmup_fraction": 0.1,
                      "weight_decay": 0.01},
        "batch_size": 16,
        "epochs": 10,
        "max_steps": None,
        "mixed_precision": "mixed_float16",
        "buckets": [128, 256, 384, 512],  # throughput only: batches of similar-length dialogues
        "pool_batches": 50,              # (shuffled pools of 50 batches), cut to a bucket
        "early_stopping": {"monitor": "val_spearman", "mode": "max", "patience": 3},
    },
    "eval": {"top_k": 3, "reference_teacher": "qwen7b"},
}
if SMOKE:
    MODEL.update(max_len=64, d_model=32, num_layers=1, num_heads=2, ffn_dim=64)
    CONFIG["max_hours"] = 0.2
    CONFIG["min_hours_to_start"] = 0.0
    CONFIG["max_units"] = 40
    CONFIG["tokenizer"]["max_items_per_source"] = 20000
    CONFIG["teacher"]["batch_dialogues"] = 16
    CONFIG["mlm"].update(seq_len=64, batch_size=4, max_steps=20, steps_per_epoch=10,
                         mixed_precision="float32", steps_per_execution=2, workers=2)
    CONFIG["mlm"]["optimizer"]["warmup_steps"] = 5
    CONFIG["student"].update(seeds=[0], batch_size=4, epochs=2, max_steps=10,
                             mixed_precision="float32", buckets=[16, 32, 48], pool_batches=2)

In [ ]:
import csv
import hashlib
import json
import math
import random
import re
import shutil
import zlib
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import islice

import numpy as np
from tokenizers import Tokenizer

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
DATA = WORK / "data"
RAW = DATA / "dialogsum"
TOK_DIR = DATA / "tokenizer"
LABELS = DATA / "labels"
CKPT = WORK / "checkpoints"
RESULTS = WORK / "results"


def json_dump(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str) + "\n")


def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def hours_left():
    return CONFIG["max_hours"] - (time.time() - SESSION_START) / 3600

## Resume
Copy a previous output of this notebook back (data, labels, checkpoints, results).

In [ ]:
PREVIOUS = []
if INPUT.exists():
    for d, _, files in os.walk(INPUT, followlinks=True):
        if f"manifest_{STAGE}.json" in files or "progress_model1_scorer.json" in files:
            PREVIOUS.append(Path(d))
for prev in PREVIOUS:
    for sub in ("data", "checkpoints", "results"):
        if (prev / sub).exists():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print("restored from", prev)

## 1. Data and tokenizer
DialogSum: one utterance per `#PersonN#:` line (tag removed, speaker kept); the 3 test rows of
a dialogue are merged; train dialogues duplicated in validation/test are dropped.
WikiText-103 raw: train articles whose title is in validation/test are dropped; text is
detokenised on read. Dialogues are written to `data/dialogsum/<split>.jsonl` (read by the
teacher script and the evaluation).

In [ ]:
SPEAKER_RE = re.compile(r"^\s*#person(\d+)#\s*:\s*", re.IGNORECASE)
TEST_ID_RE = re.compile(r"^(test_\d+)_(\d+)$")
TITLE_RE = re.compile(r"^ = (?!=)(.*?) = $")
HEADING_RE = re.compile(r"^ (= )+.*( =)+ $")
AT_RE = re.compile(r" @([-,.])@ ")
ATTACH_LEFT = {",", ".", ";", ":", "!", "?", ")", "]", "}", "%"}
CLITIC_RE = re.compile(r"^'(s|t|m|d|re|ve|ll)$", re.IGNORECASE)
ATTACH_RIGHT = {"(", "[", "{", "$", "£", "€"}


def parse_dialogue(text, max_speakers):
    """Raw dialogue -> (speaker IDs, utterances). Untagged lines continue the previous turn."""
    turns = []
    for line in text.replace("\r\n", "\n").split("\n"):
        if not line.strip():
            continue
        match = SPEAKER_RE.match(line)
        if match is None:
            if not turns:
                raise ValueError(f"Text before the first speaker tag: {line[:60]!r}")
            turns[-1][1].append(line.strip())
            continue
        speaker = int(match.group(1))
        if not 1 <= speaker <= max_speakers:
            raise ValueError(f"Speaker {speaker} outside 1..{max_speakers}")
        first = line[match.end():].strip()
        turns.append((speaker, [first] if first else []))
    kept = [(s, " ".join(parts)) for s, parts in turns if parts]
    if not kept:
        raise ValueError("Dialogue has no utterances.")
    return [s for s, _ in kept], [u for _, u in kept]


def load_dialogsum(cfg):
    from datasets import load_dataset

    ds = load_dataset(cfg["hf_path"], revision=cfg["revision"])
    out = {}
    for split in ("train", "validation", "test"):
        rows = list(ds[split])
        if split != "test":
            out[split] = []
            for r in rows:
                speakers, utterances = parse_dialogue(r["dialogue"], cfg["max_speakers"])
                out[split].append({"id": r["id"], "speakers": speakers, "utterances": utterances,
                                   "summaries": [r["summary"].strip()]})
            continue
        groups = defaultdict(list)
        for r in rows:
            m = TEST_ID_RE.match(r["id"])
            groups[m.group(1)].append((int(m.group(2)), r))
        out[split] = []
        for did in sorted(groups, key=lambda s: int(s.split("_")[1])):
            members = [r for _, r in sorted(groups[did], key=lambda kr: kr[0])]
            assert len({r["dialogue"].strip() for r in members}) == 1, did
            speakers, utterances = parse_dialogue(members[0]["dialogue"], cfg["max_speakers"])
            out[split].append({"id": did, "speakers": speakers, "utterances": utterances,
                               "summaries": [r["summary"].strip() for r in members]})
    ids = {s: {d["id"] for d in v} for s, v in out.items()}
    assert not (ids["train"] & ids["validation"] or ids["train"] & ids["test"]
                or ids["validation"] & ids["test"]), "Dialogue ID in more than one split"
    eval_texts = {"\n".join(d["utterances"]) for s in ("validation", "test") for d in out[s]}
    out["train"] = [d for d in out["train"] if "\n".join(d["utterances"]) not in eval_texts]
    return out


def article_title(line):
    match = TITLE_RE.match(line.rstrip("\n"))
    return match.group(1) if match else None


def is_heading(line):
    return bool(HEADING_RE.match(line.rstrip("\n")))


def detokenize(line):
    """' a well @-@ known " test " , isn 't it ? ' -> 'a well-known "test", isn't it?'"""
    out, glue_next, quote_open = [], False, False
    for tok in AT_RE.sub(r"\1", f" {line.strip()} ").split():
        if tok == '"':
            attach = quote_open or glue_next
            quote_open = not quote_open
            glue_after = quote_open
        else:
            attach = glue_next or tok in ATTACH_LEFT or bool(CLITIC_RE.match(tok))
            glue_after = tok in ATTACH_RIGHT
        if out and attach:
            out[-1] += tok
        else:
            out.append(tok)
        glue_next = glue_after
    return " ".join(out)


def iter_articles(lines):
    """Each article as its detokenised paragraphs (headings and blank lines skipped)."""
    paragraphs = []
    for line in lines:
        if article_title(line) is not None:
            if paragraphs:
                yield paragraphs
            paragraphs = []
        elif line.strip() and not is_heading(line):
            paragraphs.append(detokenize(line))
    if paragraphs:
        yield paragraphs


def load_wikitext(cfg):
    from datasets import load_dataset

    ds = load_dataset(cfg["hf_path"], cfg["hf_config"], revision=cfg["revision"])
    wiki = {split: ds[split]["text"] for split in ("train", "validation", "test")}
    eval_titles = {t for s in ("validation", "test") for t in map(article_title, wiki[s]) if t}
    kept, dropping = [], False
    for line in wiki["train"]:
        title = article_title(line)
        if title is not None:
            dropping = title in eval_titles
        if not dropping:
            kept.append(line)
    wiki["train"] = kept
    return wiki


NEED_DATA = not (DATA / "data_done.json").exists()
if NEED_DATA:
    DIALOGSUM = load_dialogsum(CONFIG["dialogsum"])
    if CONFIG["max_units"]:
        DIALOGSUM = {s: v[: CONFIG["max_units"]] for s, v in DIALOGSUM.items()}
    for split, records in DIALOGSUM.items():
        write_jsonl(RAW / f"{split}.jsonl", records)
    WIKI = load_wikitext(CONFIG["wikitext"])
    print({s: len(v) for s, v in DIALOGSUM.items()}, "dialogues |",
          {s: len(v) for s, v in WIKI.items()}, "WikiText lines")
else:
    DIALOGSUM = {s: read_jsonl(RAW / f"{s}.jsonl") for s in ("train", "validation", "test")}

**Tokenizer** (shared with Model 2): uncased BERT-style WordPiece, 16k vocab, fixed special IDs
`[PAD]=0 [UNK]=1 [CLS]=2 [SEP]=3 [MASK]=4 [S1]..[S8]=5..12`, deterministic alphabet and
canonical ID order. If a Model 2 notebook output is attached, its `tokenizer.json` is used.

In [ ]:
PAD_ID, UNK_ID, CLS_ID, SEP_ID, MASK_ID = 0, 1, 2, 3, 4
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + [f"[S{i}]" for i in range(1, 9)]
FIRST_REGULAR_ID = len(SPECIAL_TOKENS)


def check_special_tokens(tokenizer):
    for expected, token in enumerate(SPECIAL_TOKENS):
        assert tokenizer.token_to_id(token) == expected, f"{token} has the wrong ID"


def train_tokenizer(cfg):
    from tokenizers import BertWordPieceTokenizer
    from tokenizers.models import WordPiece

    def corpus():
        wiki = (detokenize(x) for x in WIKI["train"] if x.strip() and not is_heading(x))
        yield from islice(wiki, cfg["max_items_per_source"])
        utts = (u for d in DIALOGSUM["train"] for u in d["utterances"])
        yield from islice(utts, cfg["max_items_per_source"])

    wordpiece = BertWordPieceTokenizer(clean_text=True, handle_chinese_chars=True,
                                       strip_accents=True, lowercase=True,
                                       wordpieces_prefix=cfg["wordpieces_prefix"])
    counts = Counter()
    for text in corpus():
        counts.update(wordpiece.normalizer.normalize_str(text))
    for space in [c for c in counts if c.isspace()]:
        del counts[space]
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    alphabet = sorted(c for c, _ in ranked[: cfg["limit_alphabet"]])
    wordpiece.train_from_iterator(
        corpus(), vocab_size=cfg["vocab_size"], min_frequency=cfg["min_frequency"],
        limit_alphabet=len(alphabet), initial_alphabet=alphabet, special_tokens=SPECIAL_TOKENS,
        wordpieces_prefix=cfg["wordpieces_prefix"], show_progress=False)
    tokenizer = wordpiece._tokenizer
    rest = sorted(set(tokenizer.get_vocab()) - set(SPECIAL_TOKENS))
    ordered = {tok: i for i, tok in enumerate([*SPECIAL_TOKENS, *rest])}
    tokenizer.model = WordPiece(ordered, unk_token="[UNK]",
                                continuing_subword_prefix=cfg["wordpieces_prefix"])
    return tokenizer


if not (TOK_DIR / "tokenizer.json").exists():
    TOK_DIR.mkdir(parents=True, exist_ok=True)
    shared = []
    if INPUT.exists():
        for d, _, files in os.walk(INPUT, followlinks=True):
            if "manifest_model2_cleanup.json" in files and Path(d, "data/tokenizer/tokenizer.json").exists():
                shared.append(Path(d, "data/tokenizer"))
    if shared:
        shutil.copytree(shared[0], TOK_DIR, dirs_exist_ok=True)
        print("tokenizer from the Model 2 output:", shared[0])
    else:
        tok = train_tokenizer(CONFIG["tokenizer"])
        tok.save(str(TOK_DIR / "tokenizer.json"))
        vocab = sorted(tok.get_vocab().items(), key=lambda kv: kv[1])
        (TOK_DIR / "vocab.txt").write_text("".join(f"{t}\n" for t, _ in vocab), encoding="utf-8")
        print("tokenizer trained here")
TOKENIZER = Tokenizer.from_file(str(TOK_DIR / "tokenizer.json"))
check_special_tokens(TOKENIZER)
TOKENIZER_SHA256 = hashlib.sha256((TOK_DIR / "tokenizer.json").read_bytes()).hexdigest()
assert TOKENIZER.get_vocab_size() <= MODEL["vocab_size"]
print("tokenizer sha256", TOKENIZER_SHA256[:12], "| vocab", TOKENIZER.get_vocab_size())

**Model 1 input format** (BertSum-style): `[CLS] utt_1 [SEP] [CLS] utt_2 [SEP] …`, a speaker
ID on every token, the `[CLS]` positions index the utterances. Over `max_len`, the longest
utterances are cut first (water-filling); utterances that cannot fit are dropped and counted.

In [ ]:
def length_cap(lengths, budget):
    """Largest cap with sum(min(n, cap)) <= budget."""
    if sum(lengths) <= budget:
        return max(lengths, default=0)
    lo, hi = 0, max(lengths)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(min(n, mid) for n in lengths) <= budget:
            lo = mid
        else:
            hi = mid - 1
    return lo


def encode_dialogue(speakers, utterances, max_len):
    """-> token_ids, speaker_ids, cls_positions, truncated, dropped."""
    pieces = [e.ids for e in TOKENIZER.encode_batch(list(utterances), add_special_tokens=False)]
    keep = min(len(pieces), max_len // 2)
    cap = length_cap([len(p) for p in pieces[:keep]], max_len - 2 * keep)
    ids = np.full(max_len, PAD_ID, dtype=np.int32)
    spk = np.zeros(max_len, dtype=np.int8)
    cls = np.zeros(keep, dtype=np.int32)
    pos = truncated = 0
    for i in range(keep):
        toks = pieces[i][:cap]
        truncated += len(toks) < len(pieces[i])
        cls[i] = pos
        ids[pos:pos + len(toks) + 2] = [CLS_ID, *toks, SEP_ID]
        spk[pos:pos + len(toks) + 2] = speakers[i]
        pos += len(toks) + 2
    return ids, spk, cls, truncated, len(pieces) - keep


def pack_stream(texts, seq_len, batch=10000):
    """[CLS] + seq_len-1 tokens of the 'text [SEP] text [SEP] ...' stream per row."""
    chunks, pending = [], []
    for text in list(texts) + [None]:
        if text is not None:
            pending.append(text)
        if len(pending) == batch or (text is None and pending):
            for enc in TOKENIZER.encode_batch(pending, add_special_tokens=False):
                chunks.append(np.asarray([*enc.ids, SEP_ID], dtype=np.int32))
            pending = []
    stream = np.concatenate(chunks) if chunks else np.zeros(0, np.int32)
    body = seq_len - 1
    n = -(-len(stream) // body)
    rows = np.full((n, seq_len), PAD_ID, dtype=np.int32)
    rows[:, 0] = CLS_ID
    flat = np.full(n * body, PAD_ID, dtype=np.int32)
    flat[: len(stream)] = stream
    rows[:, 1:] = flat.reshape(n, body)
    return rows


def mlm_source(source, seq_len):
    """Rows of one MLM source: packed WikiText or one dialogue per row."""
    dataset, _, split = source.partition("_")
    if dataset == "wikitext":
        paragraphs = (p for a in islice(iter_articles(WIKI[split]), CONFIG["max_units"]) for p in a)
        ids = pack_stream(paragraphs, seq_len)
        return ids, np.zeros(ids.shape, np.int8)
    rows = [encode_dialogue(d["speakers"], d["utterances"], seq_len) for d in DIALOGSUM[split]]
    return np.stack([r[0] for r in rows]), np.stack([r[1] for r in rows])


MLM_DIR = DATA / "processed" / "mlm"
STUDENT_DIR = DATA / "processed" / "model1"
if NEED_DATA:
    mcfg = CONFIG["mlm"]
    ids, spk = [], []
    for source, copies in mcfg["corpus"].items():
        a, b = mlm_source(source, mcfg["seq_len"])
        ids.append(np.tile(a, (copies, 1)))
        spk.append(np.tile(b, (copies, 1)))
    sets = {"train": (np.concatenate(ids), np.concatenate(spk))}
    for source in mcfg["held_out"]:
        sets[source] = mlm_source(source, mcfg["seq_len"])
    for name, (a, b) in sets.items():
        (MLM_DIR / name).mkdir(parents=True, exist_ok=True)
        np.save(MLM_DIR / name / "token_ids.npy", a)
        np.save(MLM_DIR / name / "speaker_ids.npy", b)
        print(f"mlm/{name}: {len(a)} rows")

    U = CONFIG["student"]["max_utterances"]
    for split, records in DIALOGSUM.items():
        n = len(records)
        arr = {"token_ids": np.zeros((n, MODEL["max_len"]), np.int32),
               "speaker_ids": np.zeros((n, MODEL["max_len"]), np.int8),
               "cls_positions": np.zeros((n, U), np.int32), "n_utterances": np.zeros(n, np.int32)}
        counts = Counter()
        for i, r in enumerate(records):
            ids_, spk_, cls_, trunc, dropped = encode_dialogue(
                r["speakers"][:U], r["utterances"][:U], MODEL["max_len"])
            dropped += len(r["utterances"]) - len(r["utterances"][:U])
            arr["token_ids"][i], arr["speaker_ids"][i] = ids_, spk_
            arr["cls_positions"][i, :len(cls_)] = cls_
            arr["n_utterances"][i] = len(cls_)
            counts.update(truncated_dialogues=bool(trunc or dropped), truncated_utterances=trunc,
                          dropped_utterances=dropped)
        out = STUDENT_DIR / split
        out.mkdir(parents=True, exist_ok=True)
        for k, v in arr.items():
            np.save(out / f"{k}.npy", v)
        (out / "ids.json").write_text(json.dumps([r["id"] for r in records]))
        json_dump(out / "meta.json", {"dialogues": n, **counts})
        print(f"model1/{split}: {n} dialogues, {dict(counts)}")
    del WIKI
    json_dump(DATA / "data_done.json", {"tokenizer_sha256": TOKENIZER_SHA256, "config": CONFIG})
print(f"data ready after {(time.time() - SESSION_START) / 60:.1f} min")

## 2. Teacher labels (separate environment)
The script below is written to disk and run with the **teacher venv's** Python (vLLM +
PyTorch). It numbers the utterances, asks for a JSON array `[{"id", "score" 0-10, "tag"}]`,
validates it (valid JSON, every id once, integer score in range, known tag), retries once
with the invalid answer and the reason (greedy decoding would otherwise repeat itself), and
appends one line per dialogue so a cut-off run resumes. If vLLM cannot be installed it falls
back to `transformers` + `bitsandbytes` 4-bit and records that. The dialogues are split into one
shard per GPU (stable hash of the ID); each GPU labels its shard with both teachers in turn,
so no GPU idles once the small teacher is done. Shards are merged into `labels/<teacher>/` when
all are complete. In smoke mode a deterministic fake teacher replaces Qwen.

In [ ]:
TEACHER_SCRIPT = r'''
"""Teacher labelling (runs in its own environment: vLLM or transformers, never JAX/Keras)."""
import argparse, json, re, sys, time, zlib
from pathlib import Path

import numpy as np

SYSTEM = ("You write meeting notes from conversations. You rate how important each utterance "
          "is for the notes and tag what kind of content it is. You answer with JSON only.")
INSTRUCTIONS = """Rate every utterance of the conversation below.

For each utterance give:
- "id": the utterance number,
- "score": an integer from {lo} to {hi}, how important the utterance is for notes
  summarising the conversation ({lo} = irrelevant, e.g. greetings or filler; {hi} = essential),
- "tag": one of {tags}:
  - "key_point": an important fact, problem, opinion or piece of information,
  - "decision": something agreed or decided,
  - "action_item": a task, request or commitment someone will carry out,
  - "none": anything else.

Answer with a JSON array only, one object per utterance, in order, no other text:
[{{"id": 1, "score": 3, "tag": "none"}}, ...]

Conversation ({n} utterances):
{conversation}"""
RETRY = ("Your answer was not valid: {error}. Answer again with the JSON array only: exactly one "
         'object per utterance id from 1 to {n}, each with "id", an integer "score" from {lo} to '
         '{hi} and a "tag" from {tags}.')
TOO_LONG = "prompt too long for the model context"
FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


class InvalidAnswer(ValueError):
    pass


def tag_list(tags):
    return ", ".join(f'"{t}"' for t in tags)


def build_messages(record, cfg):
    lo, hi = cfg["score_range"]
    conv = "\n".join(f"{i}. #Person{s}#: {u}" for i, (s, u)
                     in enumerate(zip(record["speakers"], record["utterances"]), 1))
    user = INSTRUCTIONS.format(lo=lo, hi=hi, tags=tag_list(cfg["tags"]),
                               n=len(record["utterances"]), conversation=conv)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def as_int(v):
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return v
    if isinstance(v, float) and v.is_integer():
        return int(v)
    if isinstance(v, str) and v.strip().lstrip("-").isdigit():
        return int(v)
    return None


def validate(text, n, cfg):
    if text is None:
        raise InvalidAnswer(TOO_LONG)
    fenced = FENCE_RE.search(text)
    body = fenced.group(1) if fenced else text
    start = body.find("[")
    if start < 0:
        raise InvalidAnswer("no JSON array found")
    try:
        items, _ = json.JSONDecoder().raw_decode(body[start:])
    except json.JSONDecodeError as err:
        raise InvalidAnswer(f"invalid JSON ({err.msg})") from None
    if not isinstance(items, list):
        raise InvalidAnswer("the answer is not a JSON array")
    lo, hi = cfg["score_range"]
    by_id = {}
    for item in items:
        if not isinstance(item, dict):
            raise InvalidAnswer("every array element must be an object")
        uid, score, tag = as_int(item.get("id")), as_int(item.get("score")), item.get("tag")
        if uid is None or not 1 <= uid <= n:
            raise InvalidAnswer(f"id {item.get('id')!r} is not an utterance number 1..{n}")
        if uid in by_id:
            raise InvalidAnswer(f"id {uid} appears more than once")
        if score is None or not lo <= score <= hi:
            raise InvalidAnswer(f"id {uid}: score {item.get('score')!r} is not an integer {lo}-{hi}")
        if tag not in cfg["tags"]:
            raise InvalidAnswer(f"id {uid}: tag {tag!r} is not one of {cfg['tags']}")
        by_id[uid] = (score, tag)
    missing = [i for i in range(1, n + 1) if i not in by_id]
    if missing:
        raise InvalidAnswer(f"missing ids {missing[:10]}")
    raw = [by_id[i][0] for i in range(1, n + 1)]
    return raw, [(s - lo) / (hi - lo) for s in raw], [by_id[i][1] for i in range(1, n + 1)]


def vllm_backend(model, cfg, seed):
    from vllm import LLM, SamplingParams
    llm = LLM(model=model, dtype="half", max_model_len=cfg["max_model_len"],
              gpu_memory_utilization=cfg["gpu_memory_utilization"],
              enable_prefix_caching=cfg["enable_prefix_caching"], seed=seed)
    tok = llm.get_tokenizer()
    params = SamplingParams(temperature=cfg["temperature"], max_tokens=cfg["max_new_tokens"])

    def generate(convs):
        lengths = [len(tok.apply_chat_template(c, add_generation_prompt=True, tokenize=True))
                   for c in convs]
        ok = [i for i, n in enumerate(lengths) if n + cfg["max_new_tokens"] <= cfg["max_model_len"]]
        answers = [None] * len(convs)
        for i, out in zip(ok, llm.chat([convs[i] for i in ok], params, use_tqdm=False)):
            answers[i] = out.outputs[0].text
        return answers
    return generate


def transformers_backend(model, cfg):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(model, padding_side="left")
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16")
    lm = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant, device_map="auto")

    def generate(convs):
        answers = []
        for i in range(0, len(convs), cfg["fallback_batch_size"]):
            texts = [tok.apply_chat_template(c, add_generation_prompt=True, tokenize=False)
                     for c in convs[i:i + cfg["fallback_batch_size"]]]
            enc = tok(texts, return_tensors="pt", padding=True).to(lm.device)
            out = lm.generate(**enc, do_sample=False, max_new_tokens=cfg["max_new_tokens"])
            answers += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return answers
    return generate


def fake_backend(convs):
    """Smoke runs only: a deterministic rule instead of an LLM (never used on Kaggle)."""
    out = []
    for c in convs:
        lines = c[1]["content"].split("Conversation (")[1].split("\n")[1:]
        items = []
        for i, line in enumerate(lines, 1):
            text = line.split(": ", 1)[1].lower()
            tag = ("action_item" if " will " in f" {text} " else "decision" if "decid" in text
                   else "key_point" if len(text.split()) > 8 else "none")
            items.append({"id": i, "score": min(10, len(text.split()) // 2), "tag": tag})
        out.append(json.dumps(items))
    return out


def ids_in(*paths):
    return {json.loads(l)["id"] for p in paths if p.exists() for l in p.open() if l.strip()}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--teacher", required=True)
    ap.add_argument("--deadline", type=float, required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--num-shards", type=int, default=1)
    ap.add_argument("--backend", choices=["vllm", "transformers", "fake"], default="vllm")
    args = ap.parse_args()
    run = json.loads(Path(args.config).read_text())
    cfg, raw = run["teacher"], Path(run["raw_dir"])
    out = Path(run["labels_dir"]) / args.teacher / "shards" / str(args.shard)
    out.mkdir(parents=True, exist_ok=True)
    if args.backend == "fake":
        generate, info = fake_backend, {"backend": "fake (smoke only)", "model": None}
    elif args.backend == "transformers":
        model = cfg["fallback_models"][args.teacher]
        print(f"FALLBACK: transformers + bitsandbytes 4-bit, {model}", flush=True)
        generate = transformers_backend(model, cfg)
        info = {"backend": "transformers-bnb4 (fallback)", "model": model}
    else:
        try:
            import vllm
            generate = vllm_backend(cfg["models"][args.teacher], cfg, run["seed"])
            info = {"backend": f"vllm {vllm.__version__}", "model": cfg["models"][args.teacher]}
        except Exception as err:  # not installed, or cannot start on this GPU
            print(f"vLLM unusable: {type(err).__name__}: {err}", flush=True)
            sys.exit(3)  # the notebook installs the fallback and reruns with --backend transformers
    stats_path = out / "stats.json"
    stats = json.loads(stats_path.read_text()) if stats_path.exists() else {"splits": {}}
    stats.update(teacher=args.teacher, **info, complete=False)
    for split in cfg["splits"]:
        records = [json.loads(l) for l in (raw / f"{split}.jsonl").open() if l.strip()]
        if split == "train" and cfg["train_subset"] and cfg["train_subset"] < len(records):
            keep = set(np.random.default_rng(run["seed"]).permutation(
                sorted(r["id"] for r in records))[: cfg["train_subset"]].tolist())
            records = [r for r in records if r["id"] in keep]
        records = [r for r in records
                   if zlib.crc32(r["id"].encode()) % args.num_shards == args.shard]
        good_p, bad_p = out / f"{split}.jsonl", out / f"failed_{split}.jsonl"
        todo = [r for r in records if r["id"] not in ids_in(good_p, bad_p)]
        start = time.time()
        for b in range(0, len(todo), cfg["batch_dialogues"]):
            if time.time() > args.deadline:
                print("deadline reached, stopping (resume next session)", flush=True)
                json_out = {**stats, "stopped_early": True}
                stats_path.write_text(json.dumps(json_out, indent=2))
                return
            batch = todo[b:b + cfg["batch_dialogues"]]
            convs = [build_messages(r, cfg) for r in batch]
            answers = generate(convs)
            results, errors, attempts = {}, {}, {}
            for i, (r, a) in enumerate(zip(batch, answers)):
                try:
                    results[i] = validate(a, len(r["utterances"]), cfg)
                except InvalidAnswer as err:
                    errors[i], attempts[i] = str(err), [a]
            retry = [i for i in errors if errors[i] != TOO_LONG]
            if retry:
                lo, hi = cfg["score_range"]
                convs2 = [convs[i] + [{"role": "assistant", "content": answers[i]},
                                      {"role": "user", "content": RETRY.format(
                                          error=errors[i], n=len(batch[i]["utterances"]), lo=lo,
                                          hi=hi, tags=tag_list(cfg["tags"]))}] for i in retry]
                for i, a in zip(retry, generate(convs2)):
                    try:
                        results[i] = validate(a, len(batch[i]["utterances"]), cfg)
                        del errors[i]
                    except InvalidAnswer as err:
                        errors[i] = f"{errors[i]}; retry: {err}"
                        attempts[i].append(a)
            with good_p.open("a") as g, bad_p.open("a") as f:
                for i, r in enumerate(batch):
                    if i in errors:
                        f.write(json.dumps({"id": r["id"], "error": errors[i], "answers": attempts[i]}) + "\n")
                    else:
                        raw_s, scores, tags = results[i]
                        g.write(json.dumps({"id": r["id"], "n": len(r["utterances"]), "raw_scores": raw_s,
                                            "scores": scores, "tags": tags, "retried": i in retry}) + "\n")
            print(f"{args.teacher}[{args.shard}]/{split}: {b + len(batch)}/{len(todo)}", flush=True)
        n_good, n_bad = len(ids_in(good_p)), len(ids_in(bad_p))
        prev = stats["splits"].get(split, {})
        stats["splits"][split] = {
            "dialogues": len(records), "labelled": n_good, "failed": n_bad,
            "failure_rate": round(n_bad / max(n_good + n_bad, 1), 4),
            "seconds": round(prev.get("seconds", 0) + time.time() - start, 1),
            "labelled_this_session": len(todo),
        }
        print(args.teacher, args.shard, split, stats["splits"][split], flush=True)
    stats.update(complete=True, stopped_early=False)
    stats_path.write_text(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()
'''

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def stats_complete(path):
    return path.exists() and json.loads(path.read_text()).get("complete", False)


def teacher_done(name):
    return stats_complete(LABELS / name / "stats.json")


def shard_done(name, k):
    return stats_complete(LABELS / name / "shards" / str(k) / "stats.json")


def gpu_count():
    """GPUs visible to the driver: nvidia-smi if on PATH, else /proc (0 on CPU)."""
    if shutil.which("nvidia-smi"):
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout
        return len(out.split())
    proc = Path("/proc/driver/nvidia/gpus")
    return len(list(proc.iterdir())) if proc.exists() else 0


def merge_shards(name, n):
    """labels/<name>/shards/<k>/ -> labels/<name>/ (dialogue order, summed counts)."""
    dirs = [LABELS / name / "shards" / str(k) for k in range(n)]
    shard_stats = [json.loads((d / "stats.json").read_text()) for d in dirs]
    splits = {}
    for split in CONFIG["teacher"]["splits"]:
        order = {d["id"]: i for i, d in enumerate(DIALOGSUM[split])}
        for fname in (f"{split}.jsonl", f"failed_{split}.jsonl"):
            rows = [r for d in dirs if (d / fname).exists() for r in read_jsonl(d / fname)]
            write_jsonl(LABELS / name / fname, sorted(rows, key=lambda r: order.get(r["id"], -1)))
        parts = [s["splits"][split] for s in shard_stats]
        good, bad = sum(p["labelled"] for p in parts), sum(p["failed"] for p in parts)
        splits[split] = {"dialogues": sum(p["dialogues"] for p in parts), "labelled": good,
                         "failed": bad, "failure_rate": round(bad / max(good + bad, 1), 4),
                         "seconds": max(p["seconds"] for p in parts)}  # shards run in parallel
    json_dump(LABELS / name / "stats.json", {
        "teacher": name, "model": shard_stats[0]["model"], "backend": shard_stats[0]["backend"],
        "backends": sorted({s["backend"] for s in shard_stats}), "shards": n, "splits": splits,
        "complete": True, "stopped_early": False})


TEACHERS = list(CONFIG["teacher"]["models"])
pending = [t for t in TEACHERS if not teacher_done(t)]
print("teachers to (finish) label:", pending)
if pending:
    script = Path("/tmp/teacher_label.py")
    script.write_text(TEACHER_SCRIPT)
    run_cfg = Path("/tmp/teacher_config.json")
    run_cfg.write_text(json.dumps({"teacher": CONFIG["teacher"], "seed": CONFIG["seed"],
                                   "raw_dir": str(RAW), "labels_dir": str(LABELS)}))
    # Leave time for the other phases of this session.
    deadline = SESSION_START + 0.6 * CONFIG["max_hours"] * 3600
    # The teacher venv is built with uv: Kaggle's Python has no ensurepip (`python -m venv`
    # fails), and uv installs vLLM + torch much faster than pip.
    venv = Path("/tmp/teacher_env")
    uv = [sys.executable, "-m", "uv"]
    pip = [*uv, "pip", "install", "-q", "--python", str(venv / "bin" / "python")]
    if SMOKE:
        python, backend = sys.executable, "fake"
    else:
        if not (venv / ".ready").exists():  # a half-built venv from a failed attempt is rebuilt
            shutil.rmtree(venv, ignore_errors=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
            subprocess.run([*uv, "venv", "-q", "--python", CONFIG["teacher"]["python"], str(venv)],
                           check=True)
            subprocess.run([*pip, CONFIG["teacher"]["vllm_spec"], "numpy"])  # failure -> exit 3
            (venv / ".ready").touch()
        python, backend = str(venv / "bin" / "python"), "vllm"
    gpus = gpu_count()
    # One shard per GPU, fixed at the first session so a resume continues the same split.
    # Smoke runs use 2 shards on the CPU so the split / merge path is exercised.
    shards_file = LABELS / "shards.json"
    if not shards_file.exists():
        json_dump(shards_file, {"num_shards": 2 if SMOKE else max(gpus, 1)})
    n_shards = json.loads(shards_file.read_text())["num_shards"]
    parallel = SMOKE or gpus >= n_shards  # never two vLLM engines on one GPU
    print(f"GPUs: {gpus} | teacher shards: {n_shards} | parallel: {parallel}")

    def label(jobs, backend):
        """Run (teacher, shard) jobs, one queue per shard (= GPU); return their exit codes."""
        def queue(k):
            env = dict(os.environ, VLLM_CACHE_ROOT=f"/tmp/vllm_cache_{k}")  # no shared cache writes
            if gpus:
                env["CUDA_VISIBLE_DEVICES"] = str(k % gpus)
            codes = {}
            for teacher in [t for t, s in jobs if s == k]:
                cmd = [python, str(script), "--config", str(run_cfg), "--teacher", teacher,
                       "--shard", str(k), "--num-shards", str(n_shards),
                       "--deadline", str(deadline), "--backend", backend]
                codes[(teacher, k)] = subprocess.run(cmd, env=env).returncode
            return codes
        with ThreadPoolExecutor(n_shards if parallel else 1) as ex:
            return {job: c for codes in ex.map(queue, range(n_shards)) for job, c in codes.items()}

    codes = label([(t, k) for t in pending for k in range(n_shards) if not shard_done(t, k)],
                  backend)
    print("exit codes:", codes)
    fallback = [job for job, c in codes.items() if c == 3]  # vLLM missing or cannot run here
    if fallback:
        print("FALLBACK: installing transformers + bitsandbytes 4-bit for the teachers")
        subprocess.run([*pip, "transformers", "accelerate", "bitsandbytes", "torch", "numpy"],
                       check=True)
        print("exit codes:", label(fallback, "transformers"))
    for t in pending:
        if all(shard_done(t, k) for k in range(n_shards)):
            merge_shards(t, n_shards)
for t in TEACHERS:
    p = LABELS / t / "stats.json"
    if p.exists():
        s = json.loads(p.read_text())
        print(t, s.get("backend"), "complete" if s.get("complete") else "INCOMPLETE",
              {k: (v["labelled"], v["failed"]) for k, v in s["splits"].items()})


## 3. Oracle labels
Greedily add the utterance that most increases ROUGE-1 F1 + ROUGE-2 F1 (mean over references,
Porter stemming) of the selection; stop at no gain or 3 utterances. Selected = 1, others = 0.

In [ ]:
from rouge_score.tokenizers import DefaultTokenizer


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def rouge_n_f1(pred, ref):
    overlap = sum((pred & ref).values())
    if not overlap:
        return 0.0
    p, r = overlap / sum(pred.values()), overlap / sum(ref.values())
    return 2 * p * r / (p + r)


def oracle_record(record, cfg, tok):
    orders = [{"rouge1": 1, "rouge2": 2}[t] for t in cfg["rouge_types"]]
    tokens = [tok.tokenize(u) for u in record["utterances"]]
    refs = [{n: ngrams(tok.tokenize(r), n) for n in orders} for r in record["summaries"]]

    def objective(sel):
        toks = [t for i in sorted(sel) for t in tokens[i]]
        pred = {n: ngrams(toks, n) for n in orders}
        return float(np.mean([sum(rouge_n_f1(pred[n], ref[n]) for n in orders) for ref in refs]))

    selected, best = [], 0.0
    while len(selected) < cfg["max_utterances"]:
        gains = [(objective([*selected, i]), i) for i in range(len(tokens)) if i not in selected]
        if not gains:
            break
        score, choice = max(gains, key=lambda si: (si[0], -si[1]))
        if score <= best:
            break
        selected.append(choice)
        best = score
    selected = sorted(selected)
    return {"id": record["id"], "labels": [int(i in selected) for i in range(len(tokens))],
            "selected": selected, "objective": round(best, 6)}


if not (LABELS / "oracle" / "stats.json").exists():
    tok = DefaultTokenizer(use_stemmer=CONFIG["oracle"]["use_stemmer"])
    stats = {}
    for split, records in DIALOGSUM.items():
        rows = [oracle_record(r, CONFIG["oracle"], tok) for r in records]
        write_jsonl(LABELS / "oracle" / f"{split}.jsonl", rows)
        stats[split] = {"dialogues": len(rows),
                        "mean_selected": round(float(np.mean([len(r["selected"]) for r in rows])), 3)}
    json_dump(LABELS / "oracle" / "stats.json", {"splits": stats, "config": CONFIG["oracle"]})
print("oracle:", json.loads((LABELS / "oracle" / "stats.json").read_text())["splits"])

## 4. Model code (JAX from here on)
Layers are registered under `dialogue_models` with the same names and arguments as the
repo's `src/`, so the saved `.keras` files load there: pre-LN transformer encoder with token,
position and speaker embeddings; tied MLM head scoring only the masked positions; `[CLS]`
gathering; the student importance loss; AdamW with a per-layer learning rate.

In [ ]:
import jax
import keras
from keras import ops

# Persistent compilation cache: the 12 student runs reuse the same compiled programs.
jax.config.update("jax_compilation_cache_dir", os.environ.get("JAX_CACHE_DIR", "/tmp/jax_cache"))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)

print("keras", keras.__version__, "| jax", jax.__version__, "| devices:", jax.devices())
if ON_KAGGLE and not SMOKE:
    assert jax.devices()[0].platform == "gpu", "No GPU: Settings -> Accelerator -> GPU T4 x2."


def _init(std):
    return keras.initializers.TruncatedNormal(stddev=std)


@keras.saving.register_keras_serializable(package="dialogue_models")
class TokenAndPositionEmbedding(keras.layers.Layer):
    """Sum of token, learned position and (optionally) speaker embeddings, then dropout."""

    def __init__(self, vocab_size, max_len, d_model, num_speakers=0, dropout=0.1,
                 init_std=0.02, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size, self.max_len, self.d_model = vocab_size, max_len, d_model
        self.num_speakers, self.dropout_rate, self.init_std = num_speakers, dropout, init_std
        self.token_embedding = keras.layers.Embedding(
            vocab_size, d_model, embeddings_initializer=_init(init_std), name="token_embedding")
        self.speaker_embedding = (
            keras.layers.Embedding(num_speakers + 1, d_model,
                                   embeddings_initializer=_init(init_std), name="speaker_embedding")
            if num_speakers else None)
        self.dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape):
        self.token_embedding.build(input_shape)
        if self.speaker_embedding is not None:
            self.speaker_embedding.build(input_shape)
        self.position_embedding = self.add_weight(
            name="position_embedding", shape=(self.max_len, self.d_model),
            initializer=_init(self.init_std))
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training=None):
        length = ops.shape(token_ids)[-1]
        x = self.token_embedding(token_ids)
        x = x + ops.cast(self.position_embedding[:length], x.dtype)[None, :, :]
        if self.speaker_embedding is not None:
            if speaker_ids is None:
                raise ValueError("speaker_ids is required when num_speakers > 0")
            x = x + self.speaker_embedding(speaker_ids)
        return self.dropout(x, training=training)

    def compute_output_shape(self, input_shape):
        return (*input_shape, self.d_model)

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size, max_len=self.max_len, d_model=self.d_model,
                      num_speakers=self.num_speakers, dropout=self.dropout_rate,
                      init_std=self.init_std)
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoderBlock(keras.layers.Layer):
    """Pre-LN block: x + MHA(LN(x)), then x + FFN(LN(x)); FFN = Dense -> GELU -> Dense."""

    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1, init_std=0.02, **kwargs):
        super().__init__(**kwargs)
        if d_model % num_heads:
            raise ValueError(f"d_model={d_model} is not divisible by num_heads={num_heads}")
        self.d_model, self.num_heads, self.ffn_dim = d_model, num_heads, ffn_dim
        self.dropout_rate, self.init_std = dropout, init_std
        self.attn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="attn_norm")
        self.attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout,
            kernel_initializer=_init(init_std), name="attention")
        self.attn_dropout = keras.layers.Dropout(dropout)
        self.ffn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="ffn_norm")
        self.ffn_in = keras.layers.Dense(ffn_dim, activation="gelu",
                                         kernel_initializer=_init(init_std), name="ffn_in")
        self.ffn_out = keras.layers.Dense(d_model, kernel_initializer=_init(init_std),
                                          name="ffn_out")
        self.ffn_dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape):
        self.attn_norm.build(input_shape)
        self.attention.build(input_shape, input_shape)
        self.ffn_norm.build(input_shape)
        self.ffn_in.build(input_shape)
        self.ffn_out.build((*input_shape[:-1], self.ffn_dim))
        super().build(input_shape)

    def call(self, x, padding_mask=None, training=None):
        mask = None
        if padding_mask is not None:
            mask = ops.logical_and(padding_mask[:, :, None], padding_mask[:, None, :])
        h = self.attn_norm(x)
        h = self.attention(h, h, attention_mask=mask, training=training)
        x = x + self.attn_dropout(h, training=training)
        h = self.ffn_out(self.ffn_in(self.ffn_norm(x)))
        return x + self.ffn_dropout(h, training=training)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update(d_model=self.d_model, num_heads=self.num_heads, ffn_dim=self.ffn_dim,
                      dropout=self.dropout_rate, init_std=self.init_std)
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoder(keras.layers.Layer):
    """Embeddings, num_layers pre-LN blocks and a final LayerNorm. Token 0 is padding."""

    def __init__(self, vocab_size, max_len, d_model, num_layers, num_heads, ffn_dim,
                 dropout=0.1, init_std=0.02, num_speakers=0, pad_id=0, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size, self.max_len, self.d_model = vocab_size, max_len, d_model
        self.num_layers, self.num_heads, self.ffn_dim = num_layers, num_heads, ffn_dim
        self.dropout_rate, self.init_std = dropout, init_std
        self.num_speakers, self.pad_id = num_speakers, pad_id
        self.embedding = TokenAndPositionEmbedding(
            vocab_size, max_len, d_model, num_speakers, dropout, init_std, name="embedding")
        self.blocks = [TransformerEncoderBlock(d_model, num_heads, ffn_dim, dropout, init_std,
                                               name=f"block_{i}") for i in range(num_layers)]
        self.final_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="final_norm")

    def build(self, input_shape):
        self.embedding.build(input_shape)
        hidden = (*input_shape, self.d_model)
        for block in self.blocks:
            block.build(hidden)
        self.final_norm.build(hidden)
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training=None):
        padding_mask = ops.not_equal(token_ids, self.pad_id)
        x = self.embedding(token_ids, speaker_ids, training=training)
        for block in self.blocks:
            x = block(x, padding_mask=padding_mask, training=training)
        return self.final_norm(x)

    def compute_output_shape(self, input_shape):
        return (*input_shape, self.d_model)

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size, max_len=self.max_len, d_model=self.d_model,
                      num_layers=self.num_layers, num_heads=self.num_heads,
                      ffn_dim=self.ffn_dim, dropout=self.dropout_rate, init_std=self.init_std,
                      num_speakers=self.num_speakers, pad_id=self.pad_id)
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class MLMHead(keras.layers.Layer):
    """Gather masked positions, Dense -> GELU -> LayerNorm, then tied output + bias."""

    def __init__(self, vocab_size, d_model, init_std=0.02, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size, self.d_model, self.init_std = vocab_size, d_model, init_std
        self.transform = keras.layers.Dense(d_model, activation="gelu",
                                            kernel_initializer=_init(init_std), name="transform")
        self.norm = keras.layers.LayerNormalization(epsilon=1e-6, name="norm")

    def build(self, hidden_shape, positions_shape=None, embeddings_shape=None):
        self.transform.build(hidden_shape)
        self.norm.build(hidden_shape)
        self.bias = self.add_weight(name="bias", shape=(self.vocab_size,), initializer="zeros")
        super().build(hidden_shape)

    def call(self, hidden, positions, embeddings):
        gathered = ops.take_along_axis(hidden, ops.cast(positions, "int32")[:, :, None], axis=1)
        h = self.norm(self.transform(gathered))
        return ops.matmul(h, ops.transpose(ops.cast(embeddings, h.dtype))) + self.bias

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size, d_model=self.d_model, init_std=self.init_std)
        return config


ENCODER_KEYS = ("vocab_size", "max_len", "d_model", "num_layers", "num_heads", "ffn_dim",
                "dropout", "init_std", "num_speakers")


@keras.saving.register_keras_serializable(package="dialogue_models")
class MLMPretrainer(keras.Model):
    """Encoder + tied MLM head; inputs token_ids, speaker_ids, masked_positions."""

    def __init__(self, encoder_config, **kwargs):
        super().__init__(**kwargs)
        self.encoder_config = dict(encoder_config)
        self.encoder = TransformerEncoder(**self.encoder_config, name="encoder")
        self.head = MLMHead(self.encoder_config["vocab_size"], self.encoder_config["d_model"],
                            self.encoder_config.get("init_std", 0.02), dtype="float32",
                            name="mlm_head")

    def call(self, inputs, training=None):
        hidden = self.encoder(inputs["token_ids"], inputs["speaker_ids"], training=training)
        return self.head(hidden, inputs["masked_positions"],
                         self.encoder.embedding.token_embedding.embeddings)

    def get_config(self):
        return {**super().get_config(), "encoder_config": self.encoder_config}


@keras.saving.register_keras_serializable(package="dialogue_models")
class GatherPositions(keras.layers.Layer):
    """(hidden (batch, length, d), positions (batch, n)) -> (batch, n, d)."""

    def call(self, hidden, positions):
        return ops.take_along_axis(hidden, ops.cast(positions, "int32")[:, :, None], axis=1)

    def compute_output_shape(self, hidden_shape, positions_shape):
        return (*positions_shape, hidden_shape[-1])

    def get_config(self):
        return super().get_config()


@keras.saving.register_keras_serializable(package="dialogue_models")
class ImportanceLoss(keras.losses.Loss):
    """Per dialogue: BCE(soft targets) + lambda * KL(softmax(scale*t/T) || softmax(logit/T))."""

    def __init__(self, listwise_lambda=1.0, temperature=2.0, score_scale=10.0,
                 name="importance_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.listwise_lambda, self.temperature, self.score_scale = (
            listwise_lambda, temperature, score_scale)

    def call(self, y_true, y_pred):
        y_true, logits = ops.cast(y_true, "float32"), ops.cast(y_pred, "float32")
        mask = ops.cast(y_true >= 0, "float32")
        n = ops.maximum(ops.sum(mask, axis=-1), 1.0)
        target = ops.where(mask > 0, y_true, 0.0)
        bce = ops.sum(ops.binary_crossentropy(target, logits, from_logits=True) * mask, -1) / n
        t = self.temperature
        p_log = ops.log_softmax(ops.where(mask > 0, self.score_scale * target / t, -1e9), axis=-1)
        q_log = ops.log_softmax(ops.where(mask > 0, logits / t, -1e9), axis=-1)
        kl = ops.sum(ops.exp(p_log) * (p_log - q_log) * mask, axis=-1)
        return bce + self.listwise_lambda * kl

    def get_config(self):
        return {**super().get_config(), "listwise_lambda": self.listwise_lambda,
                "temperature": self.temperature, "score_scale": self.score_scale}


@keras.saving.register_keras_serializable(package="dialogue_models")
class LayerwiseAdamW(keras.optimizers.AdamW):
    """AdamW whose learning rate (and decoupled weight decay) is scaled per named layer."""

    def __init__(self, lr_multipliers=None, **kwargs):
        super().__init__(**kwargs)
        self.lr_multipliers = dict(lr_multipliers or {})

    def _multiplier(self, variable):
        segments = variable.path.split("/")
        return next((f for name, f in self.lr_multipliers.items() if name in segments), 1.0)

    def update_step(self, gradient, variable, learning_rate):
        return super().update_step(gradient, variable, learning_rate * self._multiplier(variable))

    def _apply_weight_decay(self, variables):
        if self.weight_decay is None:
            return
        for variable in variables:
            if self._use_weight_decay(variable):
                lr = ops.cast(self.learning_rate, variable.dtype) * self._multiplier(variable)
                variable.assign(variable - variable * ops.cast(self.weight_decay, variable.dtype) * lr)

    def get_config(self):
        return {**super().get_config(), "lr_multipliers": self.lr_multipliers}


def build_optimizer(o, total_steps, warmup):
    """LayerwiseAdamW, linear warmup then cosine decay to 0."""
    warmup = min(int(warmup), max(total_steps - 1, 0))
    if "lr_heads" in o:
        peak, multipliers = o["lr_heads"], {"encoder": o["lr_encoder"] / o["lr_heads"]}
    else:
        peak, multipliers = o["peak_lr"], {}
    schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0, decay_steps=max(total_steps - warmup, 1),
        warmup_target=peak, warmup_steps=warmup)
    opt = LayerwiseAdamW(lr_multipliers=multipliers, learning_rate=schedule,
                         weight_decay=o["weight_decay"])
    opt.exclude_from_weight_decay(var_names=["bias", "gamma", "beta", "position_embedding"])
    return opt


class TimeGuard(keras.callbacks.Callback):
    """Stop before max_hours after SESSION_START (all phases of the session share it)."""

    def __init__(self):
        super().__init__()
        self.stopped, self._longest = False, 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.time()

    def on_train_batch_end(self, batch, logs=None):
        if hours_left() <= 0:  # every call (steps_per_execution)
            self.stopped = self.model.stop_training = True

    def on_epoch_end(self, epoch, logs=None):
        self._longest = max(self._longest, time.time() - self._t)
        if hours_left() * 3600 < 1.2 * self._longest:
            print("TimeGuard: the next epoch would overrun the session, stopping.")
            self.stopped = self.model.stop_training = True


class SyncEpoch(keras.callbacks.Callback):
    """dataset.set_epoch(epoch): after a resume Keras restarts at the true epoch, not 0."""

    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_begin(self, epoch, logs=None):
        self.dataset.set_epoch(epoch)


def best_logged_value(csv_path, key, mode):
    if not csv_path.exists():
        return None
    with csv_path.open() as f:
        values = [float(r[key]) for r in csv.DictReader(f) if r.get(key)]
    return (max(values) if mode == "max" else min(values)) if values else None


def configure_devices(policy, *batch_sizes):
    keras.mixed_precision.set_dtype_policy(policy)
    n_dev = len(jax.devices())
    for bs in batch_sizes:
        assert bs % n_dev == 0, f"batch size {bs} must be divisible by the {n_dev} devices"
    if n_dev > 1:
        keras.distribution.set_distribution(keras.distribution.DataParallel())


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)

## 5. MLM pretraining
15% of the non-special tokens of each row (exact count) are chosen: 80% → `[MASK]`, 10% →
random token, 10% unchanged. Masks are drawn per batch (training) or fixed (held-out).
The step budget comes from a throughput probe (`target_hours`), stored in `plan.json` and
reused when a cut-off run resumes. `encoder.keras` is saved every epoch.

In [ ]:
def mask_batch(token_ids, cfg, vocab_size, rng):
    batch, length = token_ids.shape
    n_pred = math.ceil(cfg["mask_prob"] * length)
    inputs = token_ids.copy()
    positions = np.zeros((batch, n_pred), np.int32)
    labels = np.zeros((batch, n_pred), np.int32)
    weights = np.zeros((batch, n_pred), np.float32)
    for row in range(batch):
        eligible = np.flatnonzero(token_ids[row] >= FIRST_REGULAR_ID)
        k = min(n_pred, round(cfg["mask_prob"] * len(eligible)))
        if len(eligible) and k == 0:
            k = 1
        chosen = np.sort(rng.choice(eligible, size=k, replace=False)) if k else eligible[:0]
        positions[row, :k], labels[row, :k], weights[row, :k] = chosen, token_ids[row, chosen], 1.0
        u = rng.random(k)
        inputs[row, chosen[u < cfg["mask_token_frac"]]] = MASK_ID
        rnd = chosen[(u >= cfg["mask_token_frac"]) & (u < cfg["mask_token_frac"] + cfg["random_token_frac"])]
        inputs[row, rnd] = rng.integers(FIRST_REGULAR_ID, vocab_size, len(rnd))
    return inputs, positions, labels, weights


class MLMDataset(keras.utils.PyDataset):
    """Masked batches; training walks a fresh permutation per pass, held-out uses fixed masks."""

    def __init__(self, ids, spk, batch_size, cfg, seed=0, fixed=False, steps_per_epoch=None,
                 **kwargs):
        super().__init__(**kwargs)
        self.ids, self.spk, self.batch_size, self.cfg = ids, spk, batch_size, cfg
        self.seed, self.fixed, self.epoch = seed, fixed, 0
        n = len(ids)
        self.per_pass = math.ceil(n / batch_size) if fixed else n // batch_size
        self.steps_per_epoch = steps_per_epoch or self.per_pass
        self._orders = {}

    def __len__(self):
        return self.per_pass if self.fixed else self.steps_per_epoch

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __getitem__(self, index):
        g = index if self.fixed else self.epoch * self.steps_per_epoch + index
        if self.fixed:
            rows = np.arange(g * self.batch_size, min((g + 1) * self.batch_size, len(self.ids)))
        else:
            p, i = divmod(g, self.per_pass)
            if p not in self._orders:
                self._orders = {p: np.random.default_rng([self.seed, p]).permutation(len(self.ids))}
            rows = np.sort(self._orders[p][i * self.batch_size:(i + 1) * self.batch_size])
        n_real = len(rows)
        if n_real < self.batch_size:  # always full batches: data parallel splits them evenly
            rows = np.concatenate([rows, np.repeat(rows[-1:], self.batch_size - n_real)])
        ids = np.asarray(self.ids[rows], dtype=np.int32)
        inputs, positions, labels, weights = mask_batch(
            ids, self.cfg, MODEL["vocab_size"], np.random.default_rng([self.seed, g]))
        weights[n_real:] = 0.0  # padding rows do not count
        x = {"token_ids": inputs, "speaker_ids": np.asarray(self.spk[rows], dtype=np.int32),
             "masked_positions": positions}
        return x, labels, weights


def build_mlm_model(batch=2, n_pred=2):
    model = MLMPretrainer({k: MODEL[k] for k in ENCODER_KEYS}, name="mlm")
    model({"token_ids": ops.ones((batch, MODEL["max_len"]), "int32"),
           "speaker_ids": ops.zeros((batch, MODEL["max_len"]), "int32"),
           "masked_positions": ops.zeros((batch, n_pred), "int32")})
    return model


def encoder_model(model):
    tok = keras.Input(shape=(None,), dtype="int32", name="token_ids")
    spk = keras.Input(shape=(None,), dtype="int32", name="speaker_ids")
    return keras.Model({"token_ids": tok, "speaker_ids": spk}, model.encoder(tok, spk),
                       name="mlm_encoder")


class SaveEncoder(keras.callbacks.Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path

    def on_epoch_end(self, epoch, logs=None):
        encoder_model(self.model).save(self.path)


def compile_mlm(model, optimizer):
    model.compile(optimizer=optimizer,
                  steps_per_execution=CONFIG["mlm"]["steps_per_execution"],
                  loss=keras.losses.SparseCategoricalCrossentropy(
                      from_logits=True, reduction="mean_with_sample_weight"),
                  weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])


def load_mlm(name):
    d = MLM_DIR / name
    return np.load(d / "token_ids.npy", mmap_mode="r"), np.load(d / "speaker_ids.npy", mmap_mode="r")


MLM_OUT, MLM_RES = CKPT / "mlm", RESULTS / CONFIG["mlm"]["name"]
MLM_DONE = (MLM_RES / "metrics.json").exists()
if not MLM_DONE and hours_left() > CONFIG["min_hours_to_start"]:
    m = CONFIG["mlm"]
    set_seed(CONFIG["seed"])
    configure_devices(m["mixed_precision"], m["batch_size"])
    train_ids, train_spk = load_mlm("train")
    train_ds = MLMDataset(train_ids, train_spk, m["batch_size"], m, seed=CONFIG["seed"],
                          steps_per_epoch=m["steps_per_epoch"], workers=m["workers"])
    MLM_OUT.mkdir(parents=True, exist_ok=True)
    MLM_RES.mkdir(parents=True, exist_ok=True)
    plan_path = MLM_OUT / "plan.json"
    if plan_path.exists():
        plan = json.loads(plan_path.read_text())
    elif m["max_steps"] is not None:
        plan = {"max_steps": m["max_steps"], "source": "config"}
    else:  # throughput probe: time the second of two short epochs (the first compiles)
        probe = build_mlm_model()
        compile_mlm(probe, keras.optimizers.AdamW(learning_rate=1e-5))
        durations = []
        timer = keras.callbacks.LambdaCallback(
            on_epoch_begin=lambda e, l: durations.append(-time.time()),
            on_epoch_end=lambda e, l: durations.__setitem__(-1, durations[-1] + time.time()))
        probe.fit(MLMDataset(train_ids, train_spk, m["batch_size"], m, seed=CONFIG["seed"],
                             steps_per_epoch=m["probe_steps"], workers=m["workers"]),
                  epochs=2, callbacks=[timer], verbose=0)
        rate = m["probe_steps"] / durations[-1]
        plan = {"max_steps": int(m["target_hours"] * 3600 * rate), "source": "throughput",
                "steps_per_second": round(rate, 3), "target_hours": m["target_hours"]}
        del probe
    json_dump(plan_path, plan)
    spe = min(m["steps_per_epoch"], plan["max_steps"])
    train_ds.steps_per_epoch = spe
    epochs = math.ceil(plan["max_steps"] / spe)
    val_parts = [load_mlm(n) for n in m["held_out"]]
    val_ds = MLMDataset(np.concatenate([a for a, _ in val_parts]),
                        np.concatenate([b for _, b in val_parts]), m["batch_size"], m,
                        seed=CONFIG["seed"], fixed=True)
    set_seed(CONFIG["seed"])
    model = build_mlm_model()
    compile_mlm(model, build_optimizer(m["optimizer"], spe * epochs, m["optimizer"]["warmup_steps"]))
    print(f"MLM: {model.count_params():,} params, plan {plan}, {spe} steps/epoch x {epochs}")
    guard = TimeGuard()
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=2, callbacks=[
        SyncEpoch(train_ds), guard, SaveEncoder(MLM_OUT / "encoder.keras"),
        keras.callbacks.BackupAndRestore(str(MLM_OUT / "backup"), delete_checkpoint=False),
        keras.callbacks.CSVLogger(str(MLM_RES / "history.csv"), append=True)])
    if not guard.stopped:
        model.save(MLM_OUT / "mlm.keras")
        encoder_model(model).save(MLM_OUT / "encoder.keras")
        shutil.rmtree(MLM_OUT / "backup", ignore_errors=True)
        held_out = {}
        for name in m["held_out"]:
            a, b = load_mlm(name)
            s = model.evaluate(MLMDataset(a, b, m["batch_size"], m, seed=CONFIG["seed"], fixed=True),
                               return_dict=True, verbose=0)
            held_out[name] = {"loss": s["loss"], "accuracy": s["acc"], "rows": len(a)}
        json_dump(MLM_RES / "metrics.json", {"seed": CONFIG["seed"], "params": model.count_params(),
                                             "plan": plan, "held_out": held_out})
        MLM_DONE = True
        print("MLM held-out:", held_out)
    else:
        print("MLM interrupted: resume next session")
print("MLM done:", MLM_DONE)

## 6. Students A–E
One row per dialogue; targets per utterance (teacher score / 10 or oracle 0/1, tag index),
`-1` where there is no label; only labelled dialogues are used. Loss: BCE with soft targets +
λ · listwise KL (targets ×10, T = 2) + 0.5 · tag cross-entropy (A, B, D, E). Student E
takes its importance targets from the oracle and its tags from the 7B teacher. AdamW with lr 1e-4
for the encoder and 1e-3 for the heads, 10% warmup, cosine. Early stopping and checkpoint
selection on validation Spearman. Batches hold dialogues of similar length (shuffled pools of
50 batches, batch order shuffled) cut to the smallest of 128/256/384/512 tokens that fits:
only padding is removed, so the loss is unchanged. A student starts only when its labels (and, with MLM
init, the encoder) are ready.

In [ ]:
TAGS = ("none", "key_point", "decision", "action_item")


@dataclass
class StudentSet:
    ids: list
    token_ids: np.ndarray
    speaker_ids: np.ndarray
    cls_positions: np.ndarray
    n_utterances: np.ndarray

    @classmethod
    def load(cls, d):
        arr = {k: np.load(d / f"{k}.npy", mmap_mode="r")
               for k in ("token_ids", "speaker_ids", "cls_positions", "n_utterances")}
        return cls(ids=json.loads((d / "ids.json").read_text()), **arr)


def load_targets(source, split, data, tags_from=None):
    n, u = data.cls_positions.shape
    scores = np.full((n, u), -1, np.float32)
    tags = np.full((n, u), -1, np.int8)
    labelled = np.zeros(n, bool)
    row = {d: i for i, d in enumerate(data.ids)}
    for r in read_jsonl(LABELS / source / f"{split}.jsonl"):
        i = row.get(r["id"])
        if i is None:
            continue
        k = int(data.n_utterances[i])
        if source == "oracle":
            scores[i, :k] = r["labels"][:k]
        else:
            scores[i, :k] = r["scores"][:k]
            tags[i, :k] = [TAGS.index(t) for t in r["tags"][:k]]
        labelled[i] = True
    if tags_from:  # tags from a teacher, importance from `source` (Student E)
        tags[:] = -1
        for r in read_jsonl(LABELS / tags_from / f"{split}.jsonl"):
            i = row.get(r["id"])
            if i is not None:
                k = int(data.n_utterances[i])
                tags[i, :k] = [TAGS.index(t) for t in r["tags"][:k]]
    return scores, tags, labelled


def model_inputs(data, rows, length=None):
    """Inputs for rows, token arrays cut to length (padding only) if given."""
    return {"token_ids": np.asarray(data.token_ids[rows, :length], np.int32),
            "speaker_ids": np.asarray(data.speaker_ids[rows, :length], np.int32),
            "cls_positions": np.asarray(data.cls_positions[rows], np.int32)}


def row_lengths(token_ids):
    """Non-padding length of each row (content first, then padding)."""
    content = np.asarray(token_ids) != PAD_ID
    return np.where(content.any(1), content.shape[1] - np.argmax(content[:, ::-1], axis=1), 0)


def bucket_length(n, buckets, width):
    """Smallest bucket >= n, at most width (few distinct shapes -> few compilations)."""
    return min(next((b for b in sorted(buckets) if b >= n), width), width)


def length_batches(rows, lengths, batch_size, rng=None, pool_batches=50):
    """Batches of similar-length rows. With rng: shuffle, sort pools of pool_batches batches
    by length, shuffle the batch order. Without: sort all rows by length. Last may be partial."""
    rows = np.asarray(rows)
    if rng is None:
        order = rows[np.argsort(lengths[rows], kind="stable")]
        return [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    order, pool = rows[rng.permutation(len(rows))], batch_size * pool_batches
    batches = []
    for p in range(0, len(order), pool):
        chunk = order[p:p + pool]
        chunk = chunk[np.argsort(lengths[chunk], kind="stable")]
        batches += [chunk[i:i + batch_size] for i in range(0, len(chunk), batch_size)]
    return [batches[i] for i in rng.permutation(len(batches))]


class StudentDataset(keras.utils.PyDataset):
    """Batches of labelled dialogues; with buckets, similar lengths cut to a bucket."""

    def __init__(self, data, scores, tags, rows, batch_size, use_tags, shuffle=True, seed=0,
                 drop_last=True, max_batches=None, buckets=None, pool_batches=50, **kwargs):
        super().__init__(**kwargs)
        self.data, self.scores, self.tags, self.rows = data, scores, tags, np.asarray(rows)
        self.batch_size, self.use_tags, self.shuffle, self.seed = batch_size, use_tags, shuffle, seed
        self.drop_last, self.buckets, self.pool_batches = drop_last, buckets, pool_batches
        self.lengths = row_lengths(data.token_ids) if buckets else None
        n = len(self.rows)
        nb = n // batch_size if drop_last else math.ceil(n / batch_size)
        self.n_batches = nb if max_batches is None else min(nb, max_batches)
        self.epoch = 0
        self._batches = self._make_batches()

    def _make_batches(self):
        rng = np.random.default_rng([self.seed, self.epoch]) if self.shuffle else None
        if self.buckets:
            batches = length_batches(self.rows, self.lengths, self.batch_size, rng,
                                     self.pool_batches)
        else:
            order = self.rows if rng is None else self.rows[rng.permutation(len(self.rows))]
            batches = [order[i:i + self.batch_size] for i in range(0, len(order), self.batch_size)]
        if self.drop_last:
            batches = [b for b in batches if len(b) == self.batch_size]
        return batches[:self.n_batches]

    def set_epoch(self, epoch):
        if epoch != self.epoch:
            self.epoch = epoch
            self._batches = self._make_batches()

    def __len__(self):
        return self.n_batches

    def __getitem__(self, index):
        rows = np.sort(self._batches[index])
        length = (bucket_length(int(self.lengths[rows].max()), self.buckets,
                                self.data.token_ids.shape[1]) if self.buckets else None)
        n_real = len(rows)
        if n_real < self.batch_size:  # always full batches: data parallel splits them evenly
            rows = np.concatenate([rows, np.repeat(rows[-1:], self.batch_size - n_real)])
        y = {"importance": self.scores[rows]}
        sw = {"importance": (np.arange(len(rows)) < n_real).astype(np.float32)}
        if self.use_tags:
            t = self.tags[rows]
            y["tags"] = np.where(t < 0, 0, t).astype(np.int32)
            sw["tags"] = (t >= 0).astype(np.float32)
            sw["tags"][n_real:] = 0.0
        return model_inputs(self.data, rows, length), y, sw


def build_student(use_tags):
    init = _init(MODEL["init_std"])
    tok = keras.Input(shape=(None,), dtype="int32", name="token_ids")
    spk = keras.Input(shape=(None,), dtype="int32", name="speaker_ids")
    cls = keras.Input(shape=(None,), dtype="int32", name="cls_positions")
    enc = TransformerEncoder(**{k: MODEL[k] for k in ENCODER_KEYS}, name="encoder")
    utt = GatherPositions(name="gather_cls")(enc(tok, spk), cls)
    imp = keras.layers.Dense(1, kernel_initializer=init, dtype="float32", name="importance_logit")(utt)
    outputs = {"importance": keras.layers.Reshape((-1,), dtype="float32", name="importance")(imp)}
    if use_tags:
        outputs["tags"] = keras.layers.Dense(len(TAGS), kernel_initializer=init, dtype="float32",
                                             name="tags")(utt)
    return keras.Model({"token_ids": tok, "speaker_ids": spk, "cls_positions": cls}, outputs,
                       name="student")


def predict_utterances(model, data, rows, batch_size, buckets=None):
    """Per dialogue of rows (same order): sigmoid scores and tag names."""
    rows = np.asarray(rows)
    if buckets:
        lengths = row_lengths(data.token_ids)
        batches = length_batches(rows, lengths, batch_size)
    else:
        batches = [rows[s:s + batch_size] for s in range(0, len(rows), batch_size)]
    by_row = {}
    for batch in batches:
        length = (bucket_length(int(lengths[batch].max()), buckets, data.token_ids.shape[1])
                  if buckets else None)
        padded = np.concatenate([batch, np.repeat(batch[-1:], batch_size - len(batch))])
        pred = model.predict(model_inputs(data, padded, length), batch_size=batch_size, verbose=0)
        scores = 1.0 / (1.0 + np.exp(-np.asarray(pred["importance"], np.float64)))
        tags = np.argmax(pred["tags"], axis=-1) if "tags" in pred else None
        for j, i in enumerate(batch):
            k = int(data.n_utterances[i])
            e = {"scores": [round(float(x), 6) for x in scores[j, :k]]}
            if tags is not None:
                e["tags"] = [TAGS[t] for t in tags[j, :k]]
            by_row[int(i)] = e
    return [by_row[int(i)] for i in rows]


def spearman(p, t):
    from scipy.stats import spearmanr

    p, t = np.asarray(p, float), np.asarray(t, float)
    if len(p) < 2 or np.ptp(p) == 0 or np.ptp(t) == 0:
        return None
    return float(spearmanr(p, t).statistic)


def mean_spearman(preds, targets):
    vals = [s for p, t in zip(preds, targets) if (s := spearman(p, t)) is not None]
    return float(np.mean(vals)) if vals else 0.0


def val_spearman(model, data, scores, rows, batch_size, buckets=None):
    preds = predict_utterances(model, data, rows, batch_size, buckets)
    return mean_spearman([p["scores"] for p in preds],
                         [scores[i, :int(data.n_utterances[i])] for i in rows])


class ValidationSpearman(keras.callbacks.Callback):
    def __init__(self, *args):
        super().__init__()
        self.args = args

    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["val_spearman"] = val_spearman(self.model, *self.args)


def train_student(name, spec, seed):
    s = CONFIG["student"]
    out, res = CKPT / spec["dir"] / f"seed_{seed}", RESULTS / name / f"seed_{seed}"
    set_seed(seed)
    configure_devices(s["mixed_precision"], s["batch_size"])
    train_d, val_d = StudentSet.load(STUDENT_DIR / "train"), StudentSet.load(STUDENT_DIR / "validation")
    tr_scores, tr_tags, tr_lab = load_targets(spec["source"], "train", train_d, spec.get("tags_from"))
    va_scores, va_tags, va_lab = load_targets(spec["source"], "validation", val_d, spec.get("tags_from"))
    tr_rows, va_rows = np.flatnonzero(tr_lab), np.flatnonzero(va_lab)
    spe = len(tr_rows) // s["batch_size"]
    epochs = s["epochs"]
    if s["max_steps"]:
        spe = min(spe, s["max_steps"])
        epochs = min(epochs, math.ceil(s["max_steps"] / spe))
    buckets = s["buckets"]
    train_ds = StudentDataset(train_d, tr_scores, tr_tags, tr_rows, s["batch_size"],
                              spec["use_tags"], seed=seed, max_batches=spe, buckets=buckets,
                              pool_batches=s["pool_batches"])
    val_ds = StudentDataset(val_d, va_scores, va_tags, va_rows, s["batch_size"], spec["use_tags"],
                            shuffle=False, drop_last=False, buckets=buckets)
    model = build_student(spec["use_tags"])
    if spec["init"] == "mlm":
        pre = keras.saving.load_model(MLM_OUT / "encoder.keras").get_layer("encoder")
        model.get_layer("encoder").set_weights(pre.get_weights())
    lc = s["loss"]
    losses = {"importance": ImportanceLoss(lc["listwise_lambda"], lc["temperature"], lc["score_scale"])}
    weights = {"importance": 1.0}
    if spec["use_tags"]:
        losses["tags"] = keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction="mean_with_sample_weight")
        weights["tags"] = lc["tag_weight"]
    total = spe * epochs
    model.compile(optimizer=build_optimizer(s["optimizer"], total,
                                            s["optimizer"]["warmup_fraction"] * total),
                  loss=losses, loss_weights=weights)
    out.mkdir(parents=True, exist_ok=True)
    res.mkdir(parents=True, exist_ok=True)
    es = s["early_stopping"]
    guard = TimeGuard()
    print(f"{name} seed {seed}: {model.count_params():,} params, {len(tr_rows)} dialogues, "
          f"{spe} steps/epoch x {epochs}")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=2, callbacks=[
        SyncEpoch(train_ds), ValidationSpearman(val_d, va_scores, va_rows, s["batch_size"], buckets), guard,
        keras.callbacks.ModelCheckpoint(
            str(out / "best.keras"), monitor=es["monitor"], mode=es["mode"], save_best_only=True,
            initial_value_threshold=best_logged_value(res / "history.csv", es["monitor"], es["mode"])),
        keras.callbacks.EarlyStopping(monitor=es["monitor"], mode=es["mode"], patience=es["patience"]),
        keras.callbacks.BackupAndRestore(str(out / "backup"), delete_checkpoint=False),
        keras.callbacks.CSVLogger(str(res / "history.csv"), append=True)])
    if guard.stopped:
        return False
    shutil.rmtree(out / "backup", ignore_errors=True)
    best = keras.saving.load_model(out / "best.keras")
    test_d = StudentSet.load(STUDENT_DIR / "test")
    preds = predict_utterances(best, test_d, np.arange(len(test_d.ids)), s["batch_size"], buckets)
    write_jsonl(res / "test_predictions.jsonl",
                [{"id": i, **p} for i, p in zip(test_d.ids, preds)])
    v = val_spearman(best, val_d, va_scores, va_rows, s["batch_size"], buckets)
    json_dump(res / "metrics.json", {"seed": seed, "params": best.count_params(),
                                     "validation": {"spearman": v}})
    json_dump(res / "manifest.json", {"name": name, "seed": seed, "spec": spec, "config": s,
                                      "model": MODEL, "train_dialogues": len(tr_rows),
                                      "tokenizer_sha256": TOKENIZER_SHA256, "status": "complete"})
    print(f"{name} seed {seed}: val Spearman {v:.4f}")
    return True


def student_ready(spec):
    labels_ok = spec["source"] == "oracle" or teacher_done(spec["source"])
    labels_ok = labels_ok and (not spec.get("tags_from") or teacher_done(spec["tags_from"]))
    return labels_ok and (spec["init"] == "random" or MLM_DONE)


for name, spec in CONFIG["students"].items():
    if not student_ready(spec):
        print(f"{name}: waiting for its labels / encoder (next session)")
        continue
    for seed in CONFIG["student"]["seeds"]:
        if (RESULTS / name / f"seed_{seed}" / "metrics.json").exists():
            continue
        if hours_left() <= CONFIG["min_hours_to_start"] or not train_student(name, spec, seed):
            print("out of time for this session: resume next session")
            break

## 7. Preview results and manifest
Same inference rule for every system: top-3 utterances by score, in dialogue order, grouped
by tag. ROUGE (mean over the 3 references), retention vs the 7B teacher, agreement with the
7B teacher (Spearman, top-3 Jaccard) and decision / action-item F1 against its tags. The repo's
evaluation adds BERTScore and CPU latency.

In [ ]:
from rouge_score.rouge_scorer import RougeScorer

TAG_ORDER = ("decision", "action_item", "key_point", "none")


def top_k(scores, k):
    return sorted(sorted(range(len(scores)), key=lambda i: (-scores[i], i))[:k])


def notes_text(utts, sel, tags):
    if tags is None:
        return " ".join(utts[i] for i in sorted(sel))
    return " ".join(utts[i] for t in TAG_ORDER for i in sorted(sel) if tags[i] == t)


def evaluate_run(run, ref, ids, k):
    scorer = RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    dialogues = {d["id"]: d for d in DIALOGSUM["test"]}
    sel = {i: run[i].get("selected") or top_k(run[i]["scores"], k) for i in ids}
    ref_sel = {i: top_k(ref[i]["scores"], k) for i in ids}
    rouge = {t: [] for t in ("rouge1", "rouge2", "rougeL")}
    for i in ids:
        text = notes_text(dialogues[i]["utterances"], sel[i], run[i].get("tags"))
        s = [scorer.score(r, text) for r in dialogues[i]["summaries"]]
        for t in rouge:
            rouge[t].append(np.mean([x[t].fmeasure for x in s]))
    m = {t: float(np.mean(v)) for t, v in rouge.items()}
    pairs = [(run[i]["scores"][:len(ref[i]["scores"])], ref[i]["scores"][:len(run[i]["scores"])])
             for i in ids]
    m["spearman"] = mean_spearman([p for p, _ in pairs], [t for _, t in pairs])
    m["jaccard"] = float(np.mean([len(set(sel[i]) & set(ref_sel[i])) / max(len(set(sel[i]) | set(ref_sel[i])), 1)
                                  for i in ids]))
    if all(run[i].get("tags") for i in ids):
        for tag in ("decision", "action_item"):
            tp = fp = fn = 0
            for i in ids:
                for a, b in zip(run[i]["tags"], ref[i]["tags"]):
                    tp += a == tag and b == tag
                    fp += a == tag and b != tag
                    fn += a != tag and b == tag
            p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
            m[f"{tag}_f1"] = 2 * p * r / (p + r) if p + r else 0.0
    return m


def labels_run(path):
    return {r["id"]: {"scores": r["scores"], "tags": r.get("tags")} for r in read_jsonl(path)}


done_teachers = [t for t in TEACHERS if teacher_done(t)]
ref_name = CONFIG["eval"]["reference_teacher"]
rows = []
if ref_name in done_teachers:
    k = CONFIG["eval"]["top_k"]
    teachers = {t: labels_run(LABELS / t / "test.jsonl") for t in done_teachers}
    ref = teachers[ref_name]
    ids = sorted(set(d["id"] for d in DIALOGSUM["test"]).intersection(*teachers.values()),
                 key=lambda s: int(s.split("_")[1]))
    lead = {d["id"]: {"scores": [float(-j) for j in range(len(d["utterances"]))],
                      "selected": list(range(min(k, len(d["utterances"]))))} for d in DIALOGSUM["test"]}
    oracle = {r["id"]: {"scores": r["labels"], "selected": r["selected"]}
              for r in read_jsonl(LABELS / "oracle" / "test.jsonl")}
    systems = [("Lead-3", [lead]), ("Oracle", [oracle])]
    systems += [(f"Teacher {t}", [run]) for t, run in teachers.items()]
    for name in CONFIG["students"]:
        runs = [labels_run(p) for p in sorted((RESULTS / name).glob("seed_*/test_predictions.jsonl"))]
        if runs:
            systems.append((name, runs))
    results = {name: [evaluate_run(r, ref, ids, k) for r in runs] for name, runs in systems}
    ref_q = np.mean([results[f"Teacher {ref_name}"][0][t] for t in ("rouge1", "rouge2", "rougeL")])
    cols = ["rouge1", "rouge2", "rougeL", "retention", "spearman", "jaccard", "decision_f1",
            "action_item_f1"]
    lines = [f"# Model 1 preview ({len(ids)} test dialogues; reference teacher {ref_name})", "",
             "| System | " + " | ".join(cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for name, ms in results.items():
        for m in ms:
            m["retention"] = np.mean([m[t] for t in ("rouge1", "rouge2", "rougeL")]) / ref_q
        cells = []
        for c in cols:
            v = [m[c] for m in ms if c in m]
            cells.append("–" if not v else f"{v[0]:.3f}" if len(v) == 1
                         else f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}")
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    table = "\n".join(lines) + "\n"
    (RESULTS / "tables").mkdir(parents=True, exist_ok=True)
    (RESULTS / "tables" / "model1_preview.md").write_text(table)
    print(table)

shutil.copytree(TOK_DIR, CKPT / "model1_tokenizer", dirs_exist_ok=True)
complete = {name: sorted(int(p.parent.name.split("_")[1])
                         for p in (RESULTS / name).glob("seed_*/metrics.json"))
            for name in CONFIG["students"]}
manifest = {
    "stage": STAGE, "smoke": SMOKE, "config": CONFIG, "model": MODEL,
    "tokenizer_sha256": TOKENIZER_SHA256,
    "teachers_complete": {t: teacher_done(t) for t in TEACHERS},
    "mlm_complete": MLM_DONE, "complete_seeds": complete,
    "finished_all": all(teacher_done(t) for t in TEACHERS) and MLM_DONE
    and all(len(v) == len(CONFIG["student"]["seeds"]) for v in complete.values()),
    "device": [str(d) for d in jax.devices()],
    "session_hours": round((time.time() - SESSION_START) / 3600, 2),
    "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
json_dump(WORK / f"manifest_{STAGE}.json", manifest)
json_dump(WORK / "progress_model1_scorer.json", {k: manifest[k] for k in
          ("teachers_complete", "mlm_complete", "complete_seeds", "finished_all")})
print(json.dumps({k: v for k, v in manifest.items() if k not in ("config", "model")}, indent=2))
if not manifest["finished_all"]:
    print("Not finished: run again with this notebook's output attached to resume.")